# 09 — Backbone Comparison Analysis

**Goal:** Synthesize all results from notebooks 04-08 into master comparison tables and select the best backbones for Phase 2.

**No training.** This notebook only loads saved CSVs and generates publication-ready tables.

**Outputs:**
- `09_master_binary_comparison.csv` — All backbones + ML + FGM on all binary datasets
- `09_master_ternary_comparison.csv` — All backbones on BanglaSarc3 ternary
- `09_backbone_ranking.csv` — Final ranking with recommendations for Phase 2

In [1]:
import os, json
import numpy as np
import pandas as pd
from pathlib import Path

if os.path.exists('/kaggle/working'):
    PROJECT = Path('/kaggle/working/Sarcasm_detection')
else:
    PROJECT = Path('.').resolve()
    while not (PROJECT / '00_admin').exists() and PROJECT != PROJECT.parent:
        PROJECT = PROJECT.parent

TABLE_DIR  = PROJECT / '04_outputs' / 'tables'
REPORT_DIR = PROJECT / '04_outputs' / 'reports'
print(f"Project: {PROJECT}")

Project: /Users/sefayet/Desktop/Github/Sarcasm_detection


## 1. Load All Binary Results

In [2]:
# ── Notebook 04: ML baselines ──
ml_path = TABLE_DIR / 'baseline_ml_results.csv'
ml_df = pd.read_csv(ml_path)
print("ML baselines (nb04):")
print(ml_df.to_string(index=False))
print()

ML baselines (nb04):
           dataset  val_accuracy  val_macro_f1  val_f1_binary  test_accuracy  test_macro_f1  test_f1_binary
banglasarc3_binary      0.670823      0.670477       0.659794       0.670823       0.670692        0.664122
 banglasarc_binary      0.896282      0.887335       0.855586       0.888672       0.880616        0.849604
   ben_sarc_binary      0.642746      0.642724       0.645511       0.664587       0.664583        0.663537



In [3]:
# ── Notebook 05: BanglaBERT baselines ──
bb_path = TABLE_DIR / '05_baseline_banglabert_binary_summary_all_datasets.csv'
bb_df = pd.read_csv(bb_path)
print("BanglaBERT baselines (nb05):")
print(bb_df.to_string(index=False))
print()

BanglaBERT baselines (nb05):
     model            model_name            dataset  epochs  batch_size  learning_rate  weight_decay  warmup_ratio  max_length  seed  use_normalization      split  accuracy  precision_binary  recall_binary  f1_binary  macro_f1  n_examples
banglabert csebuetnlp/banglabert banglasarc3_binary       3           8        0.00002          0.01           0.1         128    42              False       test  0.736908          0.746114       0.718204   0.731893  0.736816         802
banglabert csebuetnlp/banglabert banglasarc3_binary       3           8        0.00002          0.01           0.1         128    42              False validation  0.766833          0.786096       0.733167   0.758710  0.766568         802
banglabert csebuetnlp/banglabert  banglasarc_binary       3           8        0.00002          0.01           0.1         128    42              False       test  0.986328          0.989637       0.974490   0.982005  0.985491         512
banglabert cseb

In [4]:
# ── Notebook 06: FGM results ──
fgm_path = TABLE_DIR / '06_fgm_banglabert_binary_summary_all_datasets.csv'
fgm_df = pd.read_csv(fgm_path)
print("BanglaBERT+FGM (nb06):")
print(fgm_df.to_string(index=False))
print()

BanglaBERT+FGM (nb06):
           dataset split  accuracy  precision_binary  recall_binary  f1_binary  macro_f1
banglasarc3_binary  test  0.769327          0.764706       0.778055   0.771323  0.769309
banglasarc3_binary   val  0.788030          0.782396       0.798005   0.790123  0.788009
 banglasarc_binary  test  0.986328          0.994764       0.969388   0.981912  0.985462
 banglasarc_binary   val  0.990215          0.994792       0.979487   0.987080  0.989603
   ben_sarc_binary  test  0.809672          0.810642       0.808112   0.809375  0.809672
   ben_sarc_binary   val  0.802262          0.794677       0.815133   0.804775  0.802229



In [5]:
# ── Notebook 07: Multi-backbone binary ──
mb_path = TABLE_DIR / '07_multi_backbone_binary_summary.csv'
mb_df = pd.read_csv(mb_path)
print("Multi-backbone binary (nb07):")
print(mb_df[['model_tag', 'dataset', 'test_accuracy', 'test_macro_f1']].to_string(index=False))
print()

Multi-backbone binary (nb07):
     model_tag            dataset  test_accuracy  test_macro_f1
   xlm-roberta    ben_sarc_binary       0.761310       0.761310
   xlm-roberta  banglasarc_binary       0.947266       0.944565
   xlm-roberta banglasarc3_binary       0.730673       0.730591
         mbert    ben_sarc_binary       0.750000       0.749611
         mbert  banglasarc_binary       0.947266       0.944762
         mbert banglasarc3_binary       0.728180       0.728010
sagorsarker-bb    ben_sarc_binary       0.751560       0.751247
sagorsarker-bb  banglasarc_binary       0.951172       0.948672
sagorsarker-bb banglasarc3_binary       0.728180       0.728164
         muril    ben_sarc_binary       0.776911       0.776165
         muril  banglasarc_binary       0.978516       0.977107
         muril banglasarc3_binary       0.735661       0.735580



## 2. Master Binary Comparison Table

In [6]:
# ── Build unified rows: model_tag, dataset, test_accuracy, test_macro_f1, test_binary_f1 ──

rows = []

# ML baselines (nb04) — need to figure out the column names
print("ML columns:", list(ml_df.columns))
for _, r in ml_df.iterrows():
    ds = None
    for c in ['dataset', 'dataset_tag', 'dataset_name']:
        if c in r.index and pd.notna(r.get(c)):
            ds = r[c]; break
    acc = None
    for c in ['test_accuracy', 'accuracy', 'val_accuracy']:
        if c in r.index and pd.notna(r.get(c)):
            acc = r[c]; break
    mf1 = None
    for c in ['test_macro_f1', 'macro_f1', 'val_macro_f1']:
        if c in r.index and pd.notna(r.get(c)):
            mf1 = r[c]; break
    if ds and mf1:
        rows.append({
            'model': 'TF-IDF + LogReg',
            'source': 'nb04',
            'dataset': ds,
            'test_accuracy': acc,
            'test_macro_f1': mf1,
        })

ML columns: ['dataset', 'val_accuracy', 'val_macro_f1', 'val_f1_binary', 'test_accuracy', 'test_macro_f1', 'test_f1_binary']


In [7]:
# BanglaBERT baselines (nb05)
print("BB columns:", list(bb_df.columns))
for _, r in bb_df.iterrows():
    ds = r.get('dataset') or r.get('dataset_tag') or r.get('dataset_name')
    acc = r.get('test_accuracy') or r.get('accuracy')
    mf1 = r.get('test_macro_f1') or r.get('macro_f1')
    if ds and mf1:
        rows.append({
            'model': 'BanglaBERT',
            'source': 'nb05',
            'dataset': ds,
            'test_accuracy': acc,
            'test_macro_f1': mf1,
        })

# BanglaBERT + FGM (nb06)
for _, r in fgm_df.iterrows():
    ds = r.get('dataset') or r.get('dataset_tag') or r.get('dataset_name')
    acc = r.get('test_accuracy') or r.get('accuracy')
    mf1 = r.get('test_macro_f1') or r.get('macro_f1')
    if ds and mf1:
        rows.append({
            'model': 'BanglaBERT + FGM',
            'source': 'nb06',
            'dataset': ds,
            'test_accuracy': acc,
            'test_macro_f1': mf1,
        })

# Multi-backbone (nb07)
tag_to_name = {
    'xlm-roberta': 'XLM-RoBERTa',
    'mbert': 'mBERT',
    'sagorsarker-bb': 'Sagorsarker BB',
    'muril': 'MuRIL',
}
for _, r in mb_df.iterrows():
    rows.append({
        'model': tag_to_name.get(r['model_tag'], r['model_tag']),
        'source': 'nb07',
        'dataset': r['dataset'],
        'test_accuracy': r['test_accuracy'],
        'test_macro_f1': r['test_macro_f1'],
    })

master_binary = pd.DataFrame(rows)
print(f"Total binary result rows: {len(master_binary)}")

BB columns: ['model', 'model_name', 'dataset', 'epochs', 'batch_size', 'learning_rate', 'weight_decay', 'warmup_ratio', 'max_length', 'seed', 'use_normalization', 'split', 'accuracy', 'precision_binary', 'recall_binary', 'f1_binary', 'macro_f1', 'n_examples']
Total binary result rows: 27


In [8]:
# ── Pivot: Macro-F1 by model x dataset ──

pivot_f1 = master_binary.pivot_table(
    index='model', columns='dataset',
    values='test_macro_f1', aggfunc='first'
)
pivot_f1['mean'] = pivot_f1.mean(axis=1)
pivot_f1 = pivot_f1.sort_values('mean', ascending=False)

print("="*90)
print("  MASTER BINARY COMPARISON — Macro-F1 (Test)")
print("="*90)
print(pivot_f1.to_string(float_format='%.4f'))
print()

# Save
master_binary.to_csv(TABLE_DIR / '09_master_binary_comparison.csv', index=False)
pivot_f1.to_csv(TABLE_DIR / '09_binary_macro_f1_pivot.csv')
print("Saved: 09_master_binary_comparison.csv, 09_binary_macro_f1_pivot.csv")

  MASTER BINARY COMPARISON — Macro-F1 (Test)
dataset           banglasarc3_binary  banglasarc_binary  ben_sarc_binary   mean
model                                                                          
BanglaBERT + FGM              0.7693             0.9855           0.8097 0.8548
BanglaBERT                    0.7368             0.9855           0.7999 0.8407
MuRIL                         0.7356             0.9771           0.7762 0.8296
XLM-RoBERTa                   0.7306             0.9446           0.7613 0.8122
Sagorsarker BB                0.7282             0.9487           0.7512 0.8094
mBERT                         0.7280             0.9448           0.7496 0.8075
TF-IDF + LogReg               0.6707             0.8806           0.6646 0.7386

Saved: 09_master_binary_comparison.csv, 09_binary_macro_f1_pivot.csv


In [9]:
# ── Pivot: Accuracy by model x dataset ──

pivot_acc = master_binary.pivot_table(
    index='model', columns='dataset',
    values='test_accuracy', aggfunc='first'
)
pivot_acc['mean'] = pivot_acc.mean(axis=1)
pivot_acc = pivot_acc.sort_values('mean', ascending=False)

print("="*90)
print("  MASTER BINARY COMPARISON — Accuracy (Test)")
print("="*90)
print(pivot_acc.to_string(float_format='%.4f'))

  MASTER BINARY COMPARISON — Accuracy (Test)
dataset           banglasarc3_binary  banglasarc_binary  ben_sarc_binary   mean
model                                                                          
BanglaBERT + FGM              0.7693             0.9863           0.8097 0.8551
BanglaBERT                    0.7369             0.9863           0.7999 0.8411
MuRIL                         0.7357             0.9785           0.7769 0.8304
XLM-RoBERTa                   0.7307             0.9473           0.7613 0.8131
Sagorsarker BB                0.7282             0.9512           0.7516 0.8103
mBERT                         0.7282             0.9473           0.7500 0.8085
TF-IDF + LogReg               0.6708             0.8887           0.6646 0.7414


## 3. Improvement Deltas

In [10]:
# ── Key comparisons ──

print("="*90)
print("  KEY IMPROVEMENT DELTAS (Macro-F1)")
print("="*90)

datasets = [c for c in pivot_f1.columns if c != 'mean']

comparisons = [
    ('BanglaBERT', 'TF-IDF + LogReg', 'Transformer vs Classical'),
    ('BanglaBERT + FGM', 'BanglaBERT', 'FGM improvement'),
    ('BanglaBERT', 'MuRIL', 'BanglaBERT vs MuRIL'),
    ('BanglaBERT', 'XLM-RoBERTa', 'BanglaBERT vs XLM-R'),
    ('BanglaBERT', 'mBERT', 'BanglaBERT vs mBERT'),
    ('MuRIL', 'mBERT', 'MuRIL vs mBERT'),
]

for better, worse, label in comparisons:
    if better in pivot_f1.index and worse in pivot_f1.index:
        deltas = []
        for ds in datasets:
            b = pivot_f1.loc[better, ds]
            w = pivot_f1.loc[worse, ds]
            if pd.notna(b) and pd.notna(w):
                deltas.append(b - w)
        if deltas:
            avg_delta = np.mean(deltas)
            print(f"  {label}: {'+' if avg_delta >= 0 else ''}{avg_delta:.4f} (avg across {len(deltas)} datasets)")

  KEY IMPROVEMENT DELTAS (Macro-F1)
  Transformer vs Classical: +0.1021 (avg across 3 datasets)
  FGM improvement: +0.0141 (avg across 3 datasets)
  BanglaBERT vs MuRIL: +0.0111 (avg across 3 datasets)
  BanglaBERT vs XLM-R: +0.0286 (avg across 3 datasets)
  BanglaBERT vs mBERT: +0.0333 (avg across 3 datasets)
  MuRIL vs mBERT: +0.0222 (avg across 3 datasets)


## 4. Ternary Results

In [11]:
# ── Load ternary results (nb08) ──

tern_path = TABLE_DIR / '08_multi_backbone_ternary_summary.csv'
if tern_path.exists():
    tern_df = pd.read_csv(tern_path)
    
    display_cols = ['model_tag', 'test_accuracy', 'test_macro_f1', 'test_weighted_f1',
                    'test_macro_precision', 'test_macro_recall']
    tern_sorted = tern_df.sort_values('test_macro_f1', ascending=False)
    
    print("="*90)
    print("  TERNARY COMPARISON — BanglaSarc3 (Test)")
    print("="*90)
    print(tern_sorted[display_cols].to_string(index=False, float_format='%.4f'))
    
    # Save
    tern_sorted.to_csv(TABLE_DIR / '09_master_ternary_comparison.csv', index=False)
    print("\nSaved: 09_master_ternary_comparison.csv")
else:
    print("Ternary results not found — run notebook 08 first.")

  TERNARY COMPARISON — BanglaSarc3 (Test)
     model_tag  test_accuracy  test_macro_f1  test_weighted_f1  test_macro_precision  test_macro_recall
    banglabert         0.6581         0.6587            0.6588                0.6596             0.6580
         muril         0.6333         0.6305            0.6306                0.6295             0.6331
         mbert         0.6093         0.6095            0.6096                0.6117             0.6090
sagorsarker-bb         0.6051         0.6024            0.6025                0.6023             0.6049
   xlm-roberta         0.6043         0.6003            0.6006                0.5994             0.6040

Saved: 09_master_ternary_comparison.csv


In [12]:
# ── Per-class F1 for ternary ──

tern_report_path = REPORT_DIR / '08_multi_backbone_ternary_classification_reports.csv'
if tern_report_path.exists():
    rpt = pd.read_csv(tern_report_path)
    label_names = ['Non-Sarcastic', 'Neutral', 'Sarcastic']
    per_class = rpt[rpt['class'].isin(label_names)]
    
    pivot_cls = per_class.pivot_table(
        index='model_tag', columns='class', values='f1-score', aggfunc='first'
    )
    if all(c in pivot_cls.columns for c in label_names):
        pivot_cls = pivot_cls[label_names]
    pivot_cls['macro_avg'] = pivot_cls.mean(axis=1)
    pivot_cls = pivot_cls.sort_values('macro_avg', ascending=False)
    
    print("="*90)
    print("  PER-CLASS F1 — BanglaSarc3 Ternary (Test)")
    print("="*90)
    print(pivot_cls.to_string(float_format='%.4f'))
    
    print()
    weakest = pivot_cls[label_names].mean().idxmin()
    strongest = pivot_cls[label_names].mean().idxmax()
    print(f"Weakest class:   {weakest} (mean F1 = {pivot_cls[label_names].mean()[weakest]:.4f})")
    print(f"Strongest class: {strongest} (mean F1 = {pivot_cls[label_names].mean()[strongest]:.4f})")

  PER-CLASS F1 — BanglaSarc3 Ternary (Test)
class           Non-Sarcastic  Neutral  Sarcastic  macro_avg
model_tag                                                   
banglabert             0.5848   0.6822     0.7090     0.6587
muril                  0.5263   0.6651     0.7000     0.6305
mbert                  0.5181   0.6446     0.6658     0.6095
sagorsarker-bb         0.5093   0.6377     0.6601     0.6024
xlm-roberta            0.4900   0.6501     0.6610     0.6003

Weakest class:   Non-Sarcastic (mean F1 = 0.5257)
Strongest class: Sarcastic (mean F1 = 0.6792)


## 5. Final Backbone Ranking & Phase 2 Recommendations

In [13]:
# ── Build final ranking ──

ranking_rows = []

# Binary mean Macro-F1
binary_means = pivot_f1['mean'].to_dict()

# Ternary Macro-F1 (map model_tag to display name)
ternary_scores = {}
if tern_path.exists():
    for _, r in tern_df.iterrows():
        tag = r['model_tag']
        name_map = {
            'banglabert': 'BanglaBERT',
            'muril': 'MuRIL',
            'xlm-roberta': 'XLM-RoBERTa',
            'mbert': 'mBERT',
            'sagorsarker-bb': 'Sagorsarker BB',
        }
        ternary_scores[name_map.get(tag, tag)] = r['test_macro_f1']

# Combine
all_models = set(list(binary_means.keys()) + list(ternary_scores.keys()))
for model in all_models:
    b = binary_means.get(model)
    t = ternary_scores.get(model)
    scores = [s for s in [b, t] if s is not None and pd.notna(s)]
    overall = np.mean(scores) if scores else None
    ranking_rows.append({
        'model': model,
        'binary_mean_f1': b,
        'ternary_f1': t,
        'overall_mean': overall,
    })

ranking_df = pd.DataFrame(ranking_rows).sort_values('overall_mean', ascending=False).reset_index(drop=True)
ranking_df['rank'] = range(1, len(ranking_df) + 1)
ranking_df = ranking_df[['rank', 'model', 'binary_mean_f1', 'ternary_f1', 'overall_mean']]

print("="*90)
print("  FINAL BACKBONE RANKING (Binary Mean + Ternary)")
print("="*90)
print(ranking_df.to_string(index=False, float_format='%.4f'))

ranking_df.to_csv(TABLE_DIR / '09_backbone_ranking.csv', index=False)
print("\nSaved: 09_backbone_ranking.csv")

  FINAL BACKBONE RANKING (Binary Mean + Ternary)
 rank            model  binary_mean_f1  ternary_f1  overall_mean
    1 BanglaBERT + FGM          0.8548         NaN        0.8548
    2       BanglaBERT          0.8407      0.6587        0.7497
    3  TF-IDF + LogReg          0.7386         NaN        0.7386
    4            MuRIL          0.8296      0.6305        0.7300
    5            mBERT          0.8075      0.6095        0.7085
    6      XLM-RoBERTa          0.8122      0.6003        0.7063
    7   Sagorsarker BB          0.8094      0.6024        0.7059

Saved: 09_backbone_ranking.csv


In [14]:
# ── Phase 2 Recommendations ──

print("="*90)
print("  PHASE 2 RECOMMENDATIONS")
print("="*90)

# Top transformer backbones (exclude ML baseline and FGM variant)
transformer_only = ranking_df[~ranking_df['model'].isin(['TF-IDF + LogReg', 'BanglaBERT + FGM'])]
top2 = transformer_only.head(2)['model'].tolist()

print(f"""
FINDINGS FROM PHASE 1:

1. BanglaBERT (csebuetnlp/banglabert) is the BEST backbone for Bengali sarcasm
   detection, winning on all 3 binary datasets and the ternary task.

2. MuRIL (google/muril-base-cased) is the SECOND-BEST backbone, consistently
   outperforming generic multilingual models (XLM-R, mBERT).

3. Language-specific pre-training matters: Bengali/Indic models (BanglaBERT,
   MuRIL) clearly beat generic multilingual models (XLM-R, mBERT, Sagorsarker).

4. FGM adversarial training on BanglaBERT gives the overall best binary results,
   confirming that robustness techniques on the best backbone work.

5. Non-Sarcastic is the hardest class in ternary classification for ALL models.

6. BanglaSarc is the easiest dataset; Ben-Sarc is the hardest.

BACKBONES FOR PHASE 2 EXPERIMENTS:
  Primary:   {top2[0]} — apply all advanced techniques here
  Secondary: {top2[1]} — apply best technique to verify generalization

PHASE 2 WILL APPLY:
  - R-Drop regularization (on {top2[0]})
  - Label Smoothing (on {top2[0]})
  - Supervised Contrastive Learning (on {top2[0]})
  - Combined FGM + best technique (on {top2[0]})
  - Best technique verification (on {top2[1]})
""")

  PHASE 2 RECOMMENDATIONS

FINDINGS FROM PHASE 1:

1. BanglaBERT (csebuetnlp/banglabert) is the BEST backbone for Bengali sarcasm
   detection, winning on all 3 binary datasets and the ternary task.

2. MuRIL (google/muril-base-cased) is the SECOND-BEST backbone, consistently
   outperforming generic multilingual models (XLM-R, mBERT).

3. Language-specific pre-training matters: Bengali/Indic models (BanglaBERT,
   MuRIL) clearly beat generic multilingual models (XLM-R, mBERT, Sagorsarker).

4. FGM adversarial training on BanglaBERT gives the overall best binary results,
   confirming that robustness techniques on the best backbone work.

5. Non-Sarcastic is the hardest class in ternary classification for ALL models.

6. BanglaSarc is the easiest dataset; Ben-Sarc is the hardest.

BACKBONES FOR PHASE 2 EXPERIMENTS:
  Primary:   BanglaBERT — apply all advanced techniques here
  Secondary: MuRIL — apply best technique to verify generalization

PHASE 2 WILL APPLY:
  - R-Drop regularizatio

In [15]:
# ── Summary statistics for thesis ──

print("="*90)
print("  THESIS-READY STATISTICS")
print("="*90)

# Count of experiments
print(f"\nTotal models compared: {len(ranking_df)}")
print(f"Total binary experiments: {len(master_binary)} (model x dataset combinations)")
if tern_path.exists():
    print(f"Total ternary experiments: {len(tern_df)}")
print(f"Total datasets: 3 binary + 1 ternary")

# Best scores per dataset
print("\nBest Macro-F1 per dataset (binary):")
for ds in [c for c in pivot_f1.columns if c != 'mean']:
    best_model = pivot_f1[ds].idxmax()
    best_score = pivot_f1[ds].max()
    print(f"  {ds}: {best_score:.4f} ({best_model})")

if tern_path.exists():
    best_tern = tern_df.loc[tern_df['test_macro_f1'].idxmax()]
    print(f"  banglasarc3_ternary: {best_tern['test_macro_f1']:.4f} ({best_tern['model_tag']})")

# Transformer gain over ML
if 'TF-IDF + LogReg' in pivot_f1.index and 'BanglaBERT + FGM' in pivot_f1.index:
    gain = pivot_f1.loc['BanglaBERT + FGM', 'mean'] - pivot_f1.loc['TF-IDF + LogReg', 'mean']
    print(f"\nBest transformer vs best ML (mean Macro-F1 gain): +{gain:.4f}")

  THESIS-READY STATISTICS

Total models compared: 7
Total binary experiments: 27 (model x dataset combinations)
Total ternary experiments: 5
Total datasets: 3 binary + 1 ternary

Best Macro-F1 per dataset (binary):
  banglasarc3_binary: 0.7693 (BanglaBERT + FGM)
  banglasarc_binary: 0.9855 (BanglaBERT)
  ben_sarc_binary: 0.8097 (BanglaBERT + FGM)
  banglasarc3_ternary: 0.6587 (banglabert)

Best transformer vs best ML (mean Macro-F1 gain): +0.1162


In [16]:
# ── Files produced ──
print("\nFiles produced by this notebook:")
for p in sorted(PROJECT.rglob('09_*')):
    if p.is_file():
        print(f"  {p.relative_to(PROJECT)}  ({p.stat().st_size / 1e3:.1f} KB)")


Files produced by this notebook:
  02_notebooks/09_backbone_comparison.ipynb  (18.6 KB)
  04_outputs/tables/09_backbone_ranking.csv  (0.5 KB)
  04_outputs/tables/09_binary_macro_f1_pivot.csv  (0.7 KB)
  04_outputs/tables/09_master_binary_comparison.csv  (2.0 KB)
  04_outputs/tables/09_master_ternary_comparison.csv  (1.8 KB)
